In [1]:
import netCDF4 as nc
import numpy as np
import os
import pandas as pd

In [23]:
from simpledbf import Dbf5
import pandas as pd

# Replace 'your_file.dbf' with the path to your .dbf file
dbf_path = r"C:\Users\HP\Documents\ArcGIS\Projects\LSTM_pre0\c13_rgi60_SpatialJoin.dbf"
dbf = Dbf5(dbf_path)
HMAdf = dbf.to_dataframe()
HMAdf.set_index("RGIId", inplace=True)
HMAdf = HMAdf.dropna(subset=['abbre'])
HMAdf

,Join_Count,TARGET_FID,GLIMSId,BgnDate,EndDate,CenLon,CenLat,O1Region,O2Region,Area,...,Area_1,station,number,Perimeter,cluster,number_1,location,四源流,sankao_1,abbre
RGIId,,,,,,,,,,,,,,,,,,,,,
RGI60-13.00002,1,1,G077951E35545N,20020802,-9999999,77.9513,35.5452,13,5,0.367,...,17400.60,卡群,21,1392.0,6,23,4,1,NaN,kq
RGI60-13.00003,1,2,G077930E35519N,20020802,-9999999,77.9295,35.5188,13,5,0.070,...,17400.60,卡群,21,1392.0,6,23,4,1,NaN,kq
RGI60-13.00004,1,3,G077924E35525N,20020802,-9999999,77.9237,35.5252,13,5,0.255,...,17400.60,卡群,21,1392.0,6,23,4,1,NaN,kq
RGI60-13.00005,1,4,G077914E35531N,20020802,-9999999,77.9141,35.5309,13,5,0.261,...,17400.60,卡群,21,1392.0,6,23,4,1,NaN,kq
RGI60-13.00006,1,5,G077914E35540N,20020802,-9999999,77.9144,35.5399,13,5,0.061,...,17400.60,卡群,21,1392.0,6,23,4,1,NaN,kq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RGI60-13.53794,1,53791,G078995E35349N,20090907,-9999999,78.9950,35.3490,13,5,0.120,...,38722.68,乌鲁瓦提,24,2141.4,2,26,5,1,喀拉喀什河,wlwt
RGI60-13.53795,1,53792,G079004E35327N,20090907,-9999999,79.0040,35.3270,13,5,0.044,...,38722.68,乌鲁瓦提,24,2141.4,2,26,5,1,喀拉喀什河,wlwt
RGI60-13.53796,1,53793,G079040E35326N,20090907,-9999999,79.0400,35.3260,13,5,0.171,...,38722.68,乌鲁瓦提,24,2141.4,2,26,5,1,喀拉喀什河,wlwt


In [2]:
# ds = nc.Dataset(r"I:\GlacierData\R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp245-Batch-1-1000.nc")

In [3]:
# RGIId = ds.variables['RGIId'][:]

In [44]:
"""This is for extract the glacier runoff data from netCDF file, which is need not to run if the csv files have already existed"""
os.chdir("I:\\GlacierData")
for scenario in ["ssp" + str(x) for x in [585]]:
    fnlist = ["R13_glac_runoff_fixed_monthly_1set_2000_2100-%s-Batch-%d-%d000.nc" % (scenario, 1000 * i + 1, i + 1) for
              i in range(
            55)] + \
             ["R14_glac_runoff_fixed_monthly_1set_2000_2100-%s-Batch-%d-%d000.nc" % (scenario, 1000 * i + 1, i + 1) for
              i in
              range(27)] + \
             ["R15_glac_runoff_fixed_monthly_1set_2000_2100-%s-Batch-%d-%d000.nc" % (scenario, 1000 * i + 1, i + 1) for
              i in
              range(12)]
    for fn in fnlist:
        print(fn)
        # fn = "R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp126-Batch-1-1000.nc"
        ds = ds1 = nc.Dataset(fn, "r")
        glacier_datas = ds.variables["glac_runoff_fixed_monthly"]  # Access the variable
        glacier_lon, glacier_lat, glacier_RGIIds = ds.variables["lon"][:], ds.variables["lat"][:], ds.variables["RGIId"]
        monthlyRuns = np.array(glacier_datas[:, :, :])
        numberGlacier = glacier_datas.shape[1]
        reshaped = monthlyRuns.reshape(12, glacier_datas.shape[1], 101* 12)
        # amonpre = np.average(reshaped, axis=2)
        # mon_peak_value_arr = np.argmax(reshaped, axis=3) # monthly peak water
        # mon_peak_month_arr =  np.max(reshaped,axis=3) # monthly peak water month
        # np.savetxt("mean_mon_pre\\m"+str(m)+fn[:-3]+".csv",delimiter=',',X=amonpre )
        for m in range(12):
            df = pd.DataFrame(np.hstack([glacier_RGIIds[:].reshape(numberGlacier, -1), reshaped[m, :, :]]))
            df = df.rename(columns={df.columns[0]: "RGIId"})
            df.to_csv(r"I:\GlacierData\glacier_mon_pre\model" + str(m) + "R131415_glac_runoff_fixed_monthly_1set_2000_2100-%s-Batch.csv"  % scenario, mode='a', header=fn[-9:]=='1-1000.nc')
    print(fn, "calculated")

R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-1-1000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-1001-2000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-2001-3000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-3001-4000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-4001-5000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-5001-6000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-6001-7000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-7001-8000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-8001-9000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-9001-10000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-10001-11000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-11001-12000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-12001-13000.nc
R13_glac_runoff_fixed_monthly_1set_2000_2100-ssp585-Batch-13001-14000.nc


In [43]:
# Zonal statistic in one model
for scenario in ['ssp126','ssp245','ssp370','ssp585']:
    glacier_df = pd.read_csv(f"I:\\GlacierData\\glacier_mon_pre\\model0R131415_glac_runoff_fixed_monthly_1set_2000_2100-{scenario}-Batch.csv", index_col="RGIId")
    HMA_glacier_df = HMAdf.merge(glacier_df, left_index=True, right_index=True, how="left")
    HMA_glacier_df = HMA_glacier_df.loc[:,[str( x) for x in range(1,1213)]+['abbre']]
    station_glacier_df = HMA_glacier_df.groupby("abbre").sum() 
    station_glacier_df=station_glacier_df.T
    # Generate a date range from January 2000 to December 2100 with monthly frequency
    time_range = pd.date_range(start='2000-01', end='2101-01', freq='ME')
    
    # Check if the number of rows in the DataFrame matches the time range
    if len(station_glacier_df) == len(time_range):
        # Add the time range as a new column
        station_glacier_df['time'] = time_range
    else:
        raise ValueError("The number of rows in the DataFrame does not match the length of the time range.")
    station_glacier_df.set_index("time", inplace=True)
    print(scenario, station_glacier_df.head())
    station_glacier_df.to_csv(f"I:\\GlacierData\\glacier_mon_pre\\R131415_glac_runoff_fixed_monthly_1set_2000_2100-{scenario}-Batch.csv")

ssp126 abbre            dsk  hsg          kq         slglk        tgzlk  wlwt  \
time                                                                     
2000-01-31  0.000000  0.0    0.000000  0.000000e+00     0.000000   0.0   
2000-02-29  0.000000  0.0    0.000000  0.000000e+00     0.000000   0.0   
2000-03-31  0.000000  0.0    0.000000  0.000000e+00     0.000000   0.0   
2000-04-30  0.000000  0.0    0.000000  0.000000e+00     0.000000   0.0   
2000-05-31 -0.004192  0.0  984.243939  2.492378e+06  5887.429979   0.0   

abbre                xhl  
time                      
2000-01-31  0.000000e+00  
2000-02-29  0.000000e+00  
2000-03-31  0.000000e+00  
2000-04-30  0.000000e+00  
2000-05-31  5.572314e+06  
ssp245 abbre            dsk  hsg           kq         slglk       tgzlk  wlwt  \
time                                                                     
2000-01-31  0.000000  0.0     0.000000  0.000000e+00     0.00000   0.0   
2000-02-29  0.000000  0.0     0.000000  0.000000e+00    

# Following script is for calculate the total mass balance in each basin

In [22]:
ds_mb = nc.Dataset(r"I:\HMAglacierData\HMA_GL_RCP_R13_multigcm_rcp45_c2_ba1_100sets_2000_2100.nc",'r')
glacier_datas = ds_mb.variables['glac_massbaltotal_monthly']  # Access the variable
glacier_RGIIds = ds_mb.variables["RGIId"]
monthlyRuns = np.array(glacier_datas)
numberGlacier = glacier_datas.shape[1]
mb_arr = np.hstack([np.array(glacier_RGIIds[:].reshape(-1,1)),glacier_datas])
mb_df = pd.DataFrame(mb_arr)
mb_df= mb_df.rename(columns={mb_df.columns[0]: "RGIId"})
mb_df.set_index("RGIId", inplace=True)
HMA_mb_df = HMAdf.merge(mb_df, left_index=True, right_index=True, how="left")
HMA_mb_df = HMA_mb_df[list(range(1,1213))+['station']]
station_mb_df = HMA_mb_df.groupby("station").sum() 

In [25]:
station_mb_df.T

station,且末,乌鲁瓦提,克勒克,克孜勒塔克,克尔古提,兰干（五）,努努买买提兰干(二),协和拉,卡拉苏,卡木鲁克,...,拜城,沙曼,沙里桂兰克,玉孜门勒克,皮山,破城子,策勒,维它克河,黄水沟,黑孜
1,7.049125,22.775442,7.076917,0.764148,0.619592,3.254531,3.645276,54.326697,2.540108,8.122467,...,1.289508,9.098578,31.654938,3.465674,2.194662,14.882404,0.607195,0.711393,2.91838,3.869051
2,4.82188,3.893309,3.086556,0.244584,0.291135,1.38224,0.923201,25.823894,0.987344,3.300458,...,0.514735,3.017735,14.370855,1.009632,0.496412,6.163862,0.086499,0.264339,1.387181,1.631859
3,5.611068,6.027147,3.044068,0.161265,0.158203,0.714063,0.972216,15.697417,0.450551,1.356727,...,0.235891,2.147642,12.270044,1.621878,0.505413,2.855031,0.082389,0.331824,0.762685,0.753583
4,6.842807,8.177481,2.892164,0.163439,0.11793,0.663314,1.330618,14.016239,0.448557,1.338246,...,0.226781,2.143521,9.430171,1.675295,0.630319,2.728854,0.105177,0.340478,0.627016,0.722096
5,9.637103,16.280994,4.519463,0.332558,0.288135,1.554499,2.504847,25.95313,0.977252,3.077188,...,0.50503,4.290067,19.390532,3.845598,1.400889,5.923401,0.225652,0.47842,1.336957,1.611416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,29.688037,82.531302,17.049862,0.884844,0.704042,3.827169,14.784572,99.388715,3.404289,12.04478,...,1.678723,13.745386,60.116444,12.312136,6.579068,26.807492,1.656246,1.631108,4.597859,4.550483
1209,19.958865,32.145573,-3.710875,-0.503774,-0.484828,-3.814526,7.361068,-59.842438,-2.214064,-5.448276,...,-0.87275,-6.837556,-35.750602,-0.496129,1.025248,-13.938627,0.364459,-0.93941,-2.771907,-5.264983
1210,-183.492187,-511.816224,-87.043729,-2.258667,-3.241864,-17.483743,-109.049285,-437.561477,-16.087201,-42.676369,...,-9.337163,-47.428746,-221.088742,-70.658574,-23.723133,-103.83301,-9.206922,-5.845983,-15.153397,-21.30728
1211,-150.617879,-488.834163,-74.022441,-2.159843,-2.800118,-15.149725,-82.718027,-433.290622,-12.952507,-37.614432,...,-7.529268,-45.239737,-221.665053,-75.012144,-24.681214,-97.393842,-8.560143,-5.048813,-13.69088,-18.04962


In [23]:
HMA_mb_df

,1,2,3,4,5,6,7,8,9,10,...,1204,1205,1206,1207,1208,1209,1210,1211,1212,station
RGIId,,,,,,,,,,,,,,,,,,,,,
RGI60-13.00001,0.005451,0.001371,0.003276,0.005648,0.01048,0.00956,0.022057,0.038798,0.049322,-0.04679,...,0.005452,0.00658,0.009729,0.018514,0.03153,0.017014,-0.201341,-0.171112,0.006202,NaN
RGI60-13.00002,0.005255,0.002929,0.005996,0.007038,0.012994,0.012748,0.021234,0.027529,0.042571,0.000066,...,0.008994,0.011595,0.013772,0.022611,0.026079,-0.047405,-0.385857,-0.386423,-0.008507,卡群
RGI60-13.00003,0.005757,0.003251,0.006648,0.007757,0.014063,0.013913,0.023025,0.030433,0.046505,-0.032252,...,0.005373,0.006882,0.007549,0.012941,0.014914,0.015451,-0.138791,-0.138146,0.009492,卡群
RGI60-13.00004,0.005775,0.003281,0.006691,0.007697,0.014558,0.014054,0.023296,0.030721,0.047385,-0.025714,...,0.008111,0.010798,0.012214,0.020413,0.023096,0.007744,-0.302715,-0.299198,0.010163,卡群
RGI60-13.00005,0.006458,0.003643,0.007491,0.008719,0.016062,0.015702,0.026011,0.034257,0.052663,-0.037409,...,0.007344,0.009557,0.0108,0.018245,0.020707,0.015644,-0.211936,-0.206422,0.011286,卡群
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RGI60-13.54427,0.010857,0.002406,0.003171,0.004926,0.006745,0.014307,0.031054,0.048621,0.070808,-0.079573,...,0.004828,0.006133,0.014887,0.029853,0.044785,0.026749,-0.503743,-0.276253,0.043897,NaN
RGI60-13.54428,0.010654,0.002394,0.003127,0.004829,0.006614,0.014046,0.030619,0.048066,0.068837,-0.093223,...,0.005234,0.006721,0.015991,0.033187,0.047566,-0.017929,-0.677874,-0.41344,0.023501,NaN
RGI60-13.54429,0.010402,0.002337,0.003033,0.004686,0.006432,0.01366,0.029512,0.046785,0.065743,-0.064148,...,0.005057,0.006416,0.015433,0.031652,0.046105,0.016394,-0.561433,-0.296255,0.030509,NaN
